# 🏋️ TD(λ) Connect-4 — N-Tuple Network Training

This notebook trains a **Temporal-Difference (TD) agent** to play Connect-4 using an **N-Tuple Network** as its value function. The agent learns entirely by **self-play** — no labelled data, no tree search during training.

### Key ideas
- **N-Tuple Network** — a lookup-table ensemble; fast, differentiable via PyTorch
- **Afterstate value function** — the network scores the board *after* a move is placed
- **Truncated TD(λ)** — bootstrapped λ-returns over a short ring buffer give a bias-variance trade-off between TD(0) and Monte Carlo
- **Parallel self-play** — thousands of games run simultaneously on a single GPU/CPU tensor

### 🗺️ Today's Journey
| Section | What happens |
|---|---|
| **§0 – Colab Setup** | Optional SSH / clone / install when running in Colab |
| **§1 – Imports** | Project components (`NTupleNetwork`, `BoardBatch`, `TrainingLogger`, …) |
| **§2 – N-Tuple Network** | Inspect and visualise the tuple set (look-up table ensemble) |
| **§3 – Evaluation (BitBully Arena)** | Round-robin tournament against α–β opponents at multiple depths |
| **§4 – Hyperparameters & Experiment Setup** | Knobs for the run; persisted to JSON for reproducibility |
| **§5 – Training Loop** | Parallel self-play + TD(λ) target + Adam + target-network Polyak update |
| **§6 – Analysis** | Adam effective per-weight learning rates and other diagnostics |
| **§7 – Main Success Factors** | What made training work |
| **§8 – Future Work** | Directions to extend the agent |

> 📝 **Prerequisites:** you've already worked through [`0_problem_illustration.ipynb`](0_problem_illustration.ipynb) for the conceptual ground — bitboard environment, afterstates, the N-tuple network, and the TD(λ) update rule.

<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/lab2/1_train_ntuple_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ⚙️ §0. Google Colab Setup

The cells below only execute when running on Google Colab.  
They set up SSH access to GitHub, clone the repo, and install the package.

> **Skip this section** when running locally — jump straight to §1.

In [ ]:
# Only relevant for Google Colab:
import pathlib
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
if False:  # Only needed for private repos
    import os

    if not pathlib.Path(os.path.expanduser("~/.ssh/id_rsa")).exists():
        !ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
    else:
        print("Key already exists!")
    !ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts 2>/dev/null
    !cat ~/.ssh/id_rsa.pub

In [ ]:
if IN_COLAB:
    if not pathlib.Path("techdays26").exists():
        !git clone https://github.com/MarkusThill/techdays26.git
    else:
        !git -C techdays26 pull
    !pip install -e techdays26[dev,lab2]

## 📦 §1. Imports

Standard library + PyTorch imports and the four main project components:

| Import | Role |
|---|---|
| `NTupleNetwork` | Value function — PyTorch `nn.Module` wrapping the LUT ensemble |
| `BoardBatch` | Vectorised Connect-4 environment (`B` games as flat tensors) |
| `best_afterstate_values` | ε-greedy afterstate search (greedy + reservoir-sampled random) |
| `TrainingLogger` | Periodic metrics logging + arena evaluation |

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

In [ ]:
import copy

import torch
import torch.nn.functional as F

from techdays26 import bitbully_arena as ba
from techdays26.bitbully_arena import get_table_legend
from techdays26.ntuple_network import NTupleNetwork
from techdays26.ntuples import NTUPLE_BITIDX_LIST_200
from techdays26.td_agent import TDConnect4AgentTorch
from techdays26.torch_board import BoardBatch
from techdays26.training import best_afterstate_values

## 🧩 §2. N-Tuple Network

An N-Tuple Network is a sum of **Look-Up Tables (LUTs)**.  
Each tuple is a fixed set of *N* board cells. For every board position the *N* cells are read out as a base-4 number (0 = empty, 1 = yellow, 2 = red, 3 = reachable), producing a **table index** that addresses one weight in the LUT.

$$V(s) = \tanh\!\left(\sum_{m=1}^{M} W_m[T_m(s)] + W_m[T_m(\mathrm{mirror}(s))]\right)$$

- **M** tuples, each of length **N** → **2 × M × 4ᴺ** parameters total  
- Mirror symmetry is exploited for free: every tuple is also evaluated on the horizontally reflected board  
- The cell below prints the tuple set and draws each tuple on a 7×6 grid

In [ ]:
from techdays26.ntuples import format_ntuple, ntuple_summary

info = ntuple_summary(NTUPLE_BITIDX_LIST_200)
print(
    f"N-tuples: {info['count']} x {info['length']}-tuples  (LUT size: {info.get('lut_size')})"
)
print(f"Hash: {info['hash'][:16]}...")
print()

for i, tup in enumerate(NTUPLE_BITIDX_LIST_200):
    print(f"--- Tuple {i:2d}: {tup} ---")
    print(format_ntuple(tup))
    print()

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

## ⚔️ §3. Evaluation — BitBully Arena

During training the agent is periodically evaluated against a suite of **BitBully** opponents — an alpha-beta minimax engine at various search depths.

The `evaluate()` function below runs a round-robin tournament via `BitBullyArena` and returns win/draw/loss counts.

In [ ]:
from bitbully import BitBully

bitbully_agent_max_depth = BitBully(opening_book="12-ply-dist", tie_break="random")

# Some weaker agents with limited depth
bitbully_agents = {}
for search_depth in (1, 2, 4, 8):
    agent = BitBully(opening_book=None, tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}ply"] = agent

for search_depth in (8, 10):
    agent = BitBully(opening_book="8-ply", tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}-ply-book8ply"] = agent

bitbully_agents["bitbully-16ply-book12ply"] = BitBully(
    opening_book="12-ply-dist", tie_break="random", max_depth=16
)
bitbully_agents["bitbully-full-strength"] = bitbully_agent_max_depth

In [ ]:
import logging


def evaluate(td_agent, rnd_agent, bitbully_agents):
    # Logger is optional; arena is silent by default unless you configure logging.
    logger = logging.getLogger("bitbully.arena")
    logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout

    agent_specs = [
        ba.AgentSpec(
            agent_id=k,
            agent=v,
            colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
            epsilons=(0.00, 0.1, 0.2, 0.3) if "full-strength" in k else (0.00,),
        )
        for k, v in bitbully_agents.items()
    ]
    matchups = (
        [ba.Matchup(yellow_id=k, red_id="ntuple") for k in bitbully_agents.keys()]
        + [ba.Matchup(yellow_id="ntuple", red_id=k) for k in bitbully_agents.keys()]
        + [
            ba.Matchup(yellow_id="ntuple", red_id="random"),
            ba.Matchup(yellow_id="random", red_id="ntuple"),
        ]
    )

    cfg = ba.ArenaConfig(
        agents=(
            *agent_specs,
            ba.AgentSpec(
                agent_id="random",
                agent=rnd_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
                epsilons=(0.00,),
            ),
            ba.AgentSpec(
                agent_id="ntuple",
                agent=td_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both,
                epsilons=(0.00,),
            ),
        ),
        n_games=50,  # n games per pairing per ε per color-swap
        time_control=ba.TimeControl(
            per_move_timeout_s=4.0,  # best-effort (measured after return)
            per_game_budget_s=45.0,  # seconds of total think time
        ),
        matchups=matchups,
        seed=None,
        log_scores=False,  # set True to also call score_all_moves() for logging
        use_tqdm=True,  # optional progress bar
        logger=logger,
    )

    # -----------------------------
    # Run arena
    # -----------------------------

    arena = ba.BitBullyArena()
    result = arena.run(cfg)
    return result

In [ ]:
from techdays26 import utils

commit_sha = utils.get_commit_hash("/content/techdays26" if IN_COLAB else ".")
reqs = utils.get_requirements_string()

## 🎛️ §4. Hyperparameters & Experiment Setup

### Key hyperparameters

| Parameter | Meaning |
|---|---|
| `B` | Batch size — number of games played in parallel |
| `epsilon` | ε-greedy exploration rate |
| `lam` | TD(λ) trace-decay (0 = TD(0), 1 = Monte Carlo) |
| `n_truncate` | Lookahead horizon for truncated λ-return |
| `tau` | Polyak coefficient for target-network EMA update |
| `lr_initial / lr_final / gamma` | Exponentially decaying Adam learning rate |

The second cell creates the output folder and persists all parameters to JSON for reproducibility.

In [ ]:
import json
import platform
import sys
from datetime import datetime
from pathlib import Path

from techdays26 import __version__ as techdays26_version
from techdays26.ntuples import ntuple_summary

# ── Device & model ─────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
pre_trained_model = None  # path to a .pt checkpoint, or None to start fresh

# ── Experiment output ──────────────────────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H-%M")
train_folder = (
    Path("/content/drive/MyDrive/models/" if IN_COLAB else "./") / f"exp_{timestamp}"
)

# ── Training ───────────────────────────────────────────────────────────────
n_steps = 25_000  # environment steps per repeat
n_evaluate = 1_000  # run arena evaluation every n_evaluate steps
n_repeats = 10  # independent runs with the same hyperparameters
B = 20_000  # parallel games (batch size)
epsilon = 0.1  # ε-greedy exploration rate
save_snapshot_steps = [
    # 1_000,
    # 10_000,
    # 20_000,
]  # save model snapshots at these steps ([] to disable)

# ── Learning-rate schedule (exponential decay) ─────────────────────────────
lr_initial = 3e-4
lr_final = 1e-6
gamma = 0.99999  # per-step decay factor

# ── Gradient clipping ──────────────────────────────────────────────────────
use_gradient_clipping = True  # clip gradient L2-norm before opt.step()
gradient_clip_max_norm = 1.0  # max global gradient norm (L2)

# ── Optimizer ──────────────────────────────────────────────────────────────
optimizer_betas = (0.9, 0.999)
optimizer_eps = 1e-8
optimizer_weight_decay = 0

# ── TD(λ) — truncated λ-return ─────────────────────────────────────────────
lam = 0.7  # trace decay  (0 = TD(0),  1 = Monte Carlo)
n_truncate = 5  # lookahead horizon

# ── Target network (Polyak / EMA) ──────────────────────────────────────────
use_target_net = True  # False → bootstrap from online net
use_online_net_for_action = True  # only relevant when use_target_net=True
tau = 0.05  # EMA coefficient (smaller = slower target update)

# ── Misc ────────────────────────────────────────────────────────────────────
use_torch_compile = True  # set False to disable torch.compile (e.g. for debugging)
use_non_losing = False  # if True, skip losing moves during action selection
save_weights = False  # if False, pass model snapshot directly (no disk I/O)
save_detailed_arena_results = False  # if False, skip per-step `step_<N>_arena_result.json` (full match details). Aggregate scores in 0_arena_metrics.json are always written.

print(f"device: {device}")

In [ ]:
# ── Create experiment folder & persist hyperparameters ────────────────────
# (reproducibility bookkeeping — not essential for understanding the algorithm)

train_folder.mkdir(parents=False, exist_ok=False)
ntuple_info = ntuple_summary(NTUPLE_BITIDX_LIST_200)

(train_folder / "0_ntuples.json").write_text(
    json.dumps({"ntuples": NTUPLE_BITIDX_LIST_200, "summary": ntuple_info}, indent=2)
)
(train_folder / "0_params.json").write_text(
    json.dumps(
        {
            "start_time": timestamp,
            "device": device,
            "pre_trained_model": str(pre_trained_model) if pre_trained_model else None,
            "n_steps": n_steps,
            "n_evaluate": n_evaluate,
            "n_repeats": n_repeats,
            "batch_size": B,
            "lr_initial": lr_initial,
            "lr_final": lr_final,
            "gamma": gamma,
            "epsilon": epsilon,
            "use_target_net": use_target_net,
            "use_online_net_for_action": use_online_net_for_action,
            "tau": tau,
            "lam": lam,
            "n_truncate": n_truncate,
            "use_torch_compile": use_torch_compile,
            "optimizer": "Adam",
            "optimizer_betas": list(optimizer_betas),
            "optimizer_eps": optimizer_eps,
            "optimizer_weight_decay": optimizer_weight_decay,
            "use_gradient_clipping": use_gradient_clipping,
            "gradient_clip_max_norm": gradient_clip_max_norm,
            "loss": "MSE",
            "td_method": f"truncated_lambda(lam={lam}, n={n_truncate})",
            "use_non_losing": use_non_losing,
            "save_detailed_arena_results": save_detailed_arena_results,
            "activation": "tanh",
            "mirror_symmetry": True,
            "n_tuples": ntuple_info["count"],
            "tuple_length": ntuple_info["length"],
            "lut_size": ntuple_info["lut_size"],
            "total_params": ntuple_info["count"] * ntuple_info["lut_size"],
            "ntuple_hash": ntuple_info["hash"],
            "commit_sha": commit_sha,
            "techdays26_version": techdays26_version,
            "python_version": sys.version,
            "pytorch_version": torch.__version__,
        },
        indent=2,
    )
)
root_log = train_folder / "0_log.txt"
root_log.write_text(
    f"techdays26 {techdays26_version} | commit {commit_sha}\n"
    f"host: {platform.node()} | python {sys.version} | torch {torch.__version__}\n"
    f"device: {device} | B={B} | n_steps={n_steps} | lr={lr_initial}→{lr_final}\n"
    f"TD(λ={lam}, n={n_truncate}) | target_net={use_target_net} | tau={tau}\n"
    + get_table_legend()
    + "\n"
)
print(f"Experiment folder: {train_folder}")
print(f"n_repeats={n_repeats}, n_steps={n_steps}, B={B}, device={device}")

## 🔁 §5. Training Loop

The loop runs `n_steps` environment steps per repeat. At each step:

1. **Store state** — snapshot the current board and target-network value into the ring buffer  
2. **Select action** — ε-greedy afterstate search: evaluate all legal moves, pick the best (or a random one)  
3. **Play move** — advance the board; record done flags and terminal rewards  
4. **Compute TD(λ) target** — fold λ-weighted bootstrapped returns back through the ring buffer  
5. **Gradient step** — MSE loss on non-random transitions; Adam + LR schedule  
6. **Polyak update** — soft-update the frozen target network toward the online network  
7. **Reset** — restart finished games in-place (no Python loop over boards)

In [ ]:
from techdays26.logger import TrainingLogger

In [ ]:
def train_one_repeat(repeat_dir, repeat_idx):
    # ── Initialise model ─────────────────────────────────────────���─────────
    torch._dynamo.reset()
    ntuple_net = (
        NTupleNetwork.load(pre_trained_model, device=device)
        if pre_trained_model
        else NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST_200).to(device)
    )
    if use_target_net:
        target_net = copy.deepcopy(ntuple_net)
        target_net.eval()
        for p in target_net.parameters():
            p.requires_grad_(False)
    else:
        target_net = ntuple_net

    if use_torch_compile:
        ntuple_net = torch.compile(ntuple_net)
        target_net = torch.compile(target_net)

    board = BoardBatch.empty(B, device)
    opt = torch.optim.Adam(ntuple_net.parameters(), lr=lr_initial)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lr_lambda=lambda step: (
            (lr_final / lr_initial) + (1 - lr_final / lr_initial) * gamma**step
        ),
    )
    logger = TrainingLogger(
        repeat_dir,
        n_evaluate,
        n_truncate,
        n_repeats,
        repeat_idx,
        evaluate_fn=lambda net_or_path: evaluate(
            TDConnect4AgentTorch(
                model_path=net_or_path if isinstance(net_or_path, str) else None,
                model=net_or_path if not isinstance(net_or_path, str) else None,
            ),
            ba.RandomAgent(),
            bitbully_agents,
        ),
        save_weights=save_weights,
        save_snapshot_steps=save_snapshot_steps,
        save_detailed_arena_results=save_detailed_arena_results,
    )

    # ── Ring buffer (n_truncate + 1 slots) ────────────────────────────────
    buf_size = n_truncate + 1
    buf_all = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_active = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_moves = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_V = torch.zeros((buf_size, B), dtype=torch.float32, device=device)
    buf_done = torch.zeros((buf_size, B), dtype=torch.bool, device=device)
    buf_reward = torch.zeros((buf_size, B), dtype=torch.float32, device=device)
    buf_random = torch.zeros((buf_size, B), dtype=torch.bool, device=device)

    loss = torch.tensor(float("nan"))
    V_pred = torch.zeros(B, device=device)
    update_mask = torch.zeros(B, dtype=torch.bool, device=device)

    for step in range(n_steps + 1):
        idx = step % buf_size

        # ── 1. Store current state ─────────────────────────────────────────
        buf_all[idx] = board.all_tokens
        buf_active[idx] = board.active_tokens
        buf_moves[idx] = board.moves_left
        with torch.no_grad():
            buf_V[idx] = target_net(board).float()

        # ── 2. Select action — ε-greedy afterstate search ─────────────────
        randomize = torch.rand((B,), device=device) < epsilon
        buf_random[idx] = randomize
        with torch.no_grad():
            best_mv, _ = best_afterstate_values(
                board,
                ntuple_net if use_online_net_for_action else target_net,
                randomize=randomize,
                use_non_losing=False,
            )

        # ── 3. Play move, record done & reward ────────────────────────────
        board.play_masks(best_mv)
        done = board.done()
        buf_done[idx] = done
        buf_reward[idx] = torch.where(
            done, board.reward().float(), torch.zeros(B, device=device)
        )

        # ── 4. Compute truncated λ-return and gradient step ───────────────
        if step >= n_truncate:
            oldest = (step - n_truncate) % buf_size

            with torch.no_grad():
                # Bootstrap from current value or terminal reward
                G = torch.where(done, board.reward().float(), target_net(board).float())
                # Fold in λ-weighted returns from oldest → newest
                for k in range(n_truncate - 1, -1, -1):
                    bk, bk1 = (oldest + k) % buf_size, (oldest + k + 1) % buf_size
                    G = torch.where(
                        buf_done[bk], buf_reward[bk], (1 - lam) * buf_V[bk1] + lam * G
                    )

            oldest_board = BoardBatch(
                buf_all[oldest], buf_active[oldest], buf_moves[oldest]
            )
            V_pred = ntuple_net(oldest_board)
            update_mask = buf_done[oldest] | ~buf_random[oldest]

            loss = F.mse_loss(V_pred[update_mask], G[update_mask])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            logger.snapshot_weights(ntuple_net, step)  # capture ΔW before update
            if use_gradient_clipping:
                torch.nn.utils.clip_grad_norm_(
                    ntuple_net.parameters(), max_norm=gradient_clip_max_norm
                )
            opt.step()
            scheduler.step()

        # ── 5. Polyak-update target network ───────────────────────────────
        if use_target_net:
            with torch.no_grad():
                for p_t, p_o in zip(target_net.parameters(), ntuple_net.parameters()):
                    p_t.data.mul_(1 - tau).add_(p_o.data, alpha=tau)

        # ── 6. Log metrics & run arena evaluation ─────────────────────────
        if step >= n_truncate:
            logger(
                step, ntuple_net, opt, loss, done, randomize, update_mask, V_pred, board
            )

        # ── 7. Reset finished games ────────────────────────────────────────
        board.reset(done)

    return ntuple_net

In [ ]:
for repeat_idx in range(n_repeats):
    repeat_dir = (
        train_folder / f"repeat_{repeat_idx}" if n_repeats > 1 else train_folder
    )
    if n_repeats > 1:
        repeat_dir.mkdir(parents=False, exist_ok=False)
        print(
            f"\n{'=' * 80}\n=== Repeat {repeat_idx + 1}/{n_repeats} ===\n{'=' * 80}\n"
        )
    ntuple_net = train_one_repeat(repeat_dir, repeat_idx)

print(f"\nAll {n_repeats} repeat(s) completed.")

## 🔬 §6. Analysis

Visualise the Adam optimiser's **effective per-weight learning rates** after training.  
Because Adam adapts each weight individually, the actual update magnitude varies widely — this plot reveals how much the adaptive scaling has compressed or amplified the scheduled LR.

In [ ]:
# from techdays26.plots import plot_adam_effective_lr

# plot_adam_effective_lr(opt)

## 🏅 §7. Main Success Factors

A handful of design choices made the difference between a network that stalls and one that actually reaches α-β-10-ply-level play:

- **Afterstate value function** — learning V on the board *after* the move, not on (s, a), halves the state space and gives cleaner targets.
- **Target network + Polyak update** (`τ = 0.05`) — decouples the bootstrap target from the weight under update; prevents oscillation.
- **Truncated TD(λ)** (`λ = 0.7`, `n = 5`) — intermediate between TD(0) and Monte Carlo; lower variance than MC, lower bias than TD(0).
- **Mirror symmetry** — every tuple is evaluated on both the board and its reflection, doubling effective data with zero extra parameters.
- **Parallel self-play** (`B = 20 000`) — batch-first design keeps the GPU saturated; gradient noise from many independent games stabilises learning.
- **Adam with LR decay** (`3e-4 → 1e-6`) — aggressive early, fine at the end; per-weight scaling handles the heavy-tailed LUT update distribution.
- **ε-greedy (0.1) with action selection from the online net** — enough exploration to escape local optima without destabilising the target.

## 🚀 §8. Future Work

- **Deeper networks** — stacking multiple N-Tuple layers or adding hidden layers
- **CNNs** — convolutional feature extraction on the 6×7 board
- **Step-size adaptation** — custom per-weight learning-rate schedules beyond Adam
- **Real backward view** of eligibility traces (replacing the current forward-view approximation)

---

### ✅ Summary

You trained an N-Tuple Network Connect-4 agent with truncated TD(λ) and ε-greedy self-play. The training loop batched thousands of games in parallel, bootstrapped from a Polyak-averaged target network, and evaluated the agent against BitBully α–β opponents periodically.

> 📊 **Next step:** head to [`2_plot_metrics.ipynb`](2_plot_metrics.ipynb) to compare learning curves and arena results across multiple runs.